In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader
from math import cos , sin
import pandas as pd

In [2]:
class Encoder(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.embed_dim = embed_dim

    self.num_heads = 4
    self.mat_size = self.embed_dim//self.num_heads


    self.querym1 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym1 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem1 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.querym2 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym2 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem2 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.querym3 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym3 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem3 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.querym4 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym4 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem4 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.final_weight = nn.Parameter(torch.rand(self.embed_dim,self.embed_dim))

    self.linear = nn.Sequential(
        nn.Linear(embed_dim,embed_dim*4),
        nn.ReLU(),
        nn.Linear(embed_dim*4,embed_dim)
    )

    self.layernorm1 = nn.LayerNorm(embed_dim)
    self.layernorm2 = nn.LayerNorm(embed_dim)

  def multihead(self,embedding):

    #MULTI ATTENTION
    queryA = torch.matmul(embedding, self.querym1)
    keyA =  torch.matmul(embedding, self.keym1)
    valueA = torch.matmul(embedding, self.valuem1)

    queryB = torch.matmul(embedding, self.querym2)
    keyB =  torch.matmul(embedding, self.keym2)
    valueB = torch.matmul(embedding, self.valuem2)

    queryC = torch.matmul(embedding, self.querym3)
    keyC =  torch.matmul(embedding, self.keym3)
    valueC = torch.matmul(embedding, self.valuem3)

    queryD = torch.matmul(embedding, self.querym4)
    keyD =  torch.matmul(embedding, self.keym4)
    valueD = torch.matmul(embedding, self.valuem4)

    #final outputs of each attention

    weightA = torch.matmul(queryA,keyA.transpose(-2,-1))
    weightA = weightA/(self.mat_size**0.5)
    weightA = torch.softmax(weightA,dim = -1)
    embedA = torch.matmul(weightA,valueA)


    weightB = torch.matmul(queryB,keyB.transpose(-2,-1))
    weightB = weightB/(self.mat_size**0.5)
    weightB = torch.softmax(weightB,dim = -1)
    embedB = torch.matmul(weightB,valueB)

    weightC = torch.matmul(queryC,keyC.transpose(-2,-1))
    weightC = weightC/(self.mat_size**0.5)
    weightC = torch.softmax(weightC,dim = -1)
    embedC = torch.matmul(weightC,valueC)

    weightD = torch.matmul(queryD,keyD.transpose(-2,-1))
    weightD = weightD/(self.mat_size**0.5)
    weightD = torch.softmax(weightD,dim = -1)
    embedD = torch.matmul(weightD,valueD)


    final_embed = torch.cat([embedA,embedB,embedC,embedD],dim = -1)
    final_embed = torch.matmul(final_embed,self.final_weight)

    return final_embed

  def forward(self,embedding):

    final_embed = self.multihead(embedding)

    add = embedding + final_embed

    normmat = self.layernorm1(add)

    linear = self.linear(normmat)

    final_embed = normmat + linear

    norm_final = self.layernorm2(final_embed)

    return norm_final

class Decoder(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.embed_dim = embed_dim

    self.num_heads = 4
    self.mat_size = self.embed_dim//self.num_heads


    self.querym1 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym1 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem1 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.querym2 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym2 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem2 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.querym3 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym3 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem3 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.querym4 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.keym4 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))
    self.valuem4 = nn.Parameter(torch.rand(self.embed_dim,self.mat_size,requires_grad=True))

    self.final_weight = nn.Parameter(torch.rand(self.embed_dim,self.embed_dim))

    self.linear = nn.Sequential(
        nn.Linear(embed_dim,embed_dim*4),
        nn.ReLU(),
        nn.Linear(embed_dim*4,embed_dim)
    )

    self.layernorm1 = nn.LayerNorm(embed_dim)
    self.layernorm2 = nn.LayerNorm(embed_dim)
    self.layernorm3 = nn.LayerNorm(embed_dim)

  def masking(self,matrix):

    a = matrix.size(0)
    b = matrix.size(1)
    c = matrix.size(2)
    output = torch.zeros_like(matrix)

    for i in range(a):
      for j in range(b):
        for k in range(c):
          if(k>j):
            output[i][j][k] = float('-inf')
    return output

  def masked_multihead(self,embedding):

    #MULTI ATTENTION
    queryA = torch.matmul(embedding, self.querym1)
    keyA =  torch.matmul(embedding, self.keym1)
    valueA = torch.matmul(embedding, self.valuem1)

    queryB = torch.matmul(embedding, self.querym2)
    keyB =  torch.matmul(embedding, self.keym2)
    valueB = torch.matmul(embedding, self.valuem2)

    queryC = torch.matmul(embedding, self.querym3)
    keyC =  torch.matmul(embedding, self.keym3)
    valueC = torch.matmul(embedding, self.valuem3)

    queryD = torch.matmul(embedding, self.querym4)
    keyD =  torch.matmul(embedding, self.keym4)
    valueD = torch.matmul(embedding, self.valuem4)


    weightA = torch.matmul(queryA,keyA.transpose(-2,-1))
    weightA = weightA/(self.mat_size**0.5)
    weightA += self.masking(weightA)
    weightA = torch.softmax(weightA,dim = -1)
    embedA = torch.matmul(weightA,valueA)


    weightB = torch.matmul(queryB,keyB.transpose(-2,-1))
    weightB = weightB/(self.mat_size**0.5)
    weightB += self.masking(weightB)
    weightB = torch.softmax(weightB,dim = -1)
    embedB = torch.matmul(weightB,valueB)

    weightC = torch.matmul(queryC,keyC.transpose(-2,-1))
    weightC = weightC/(self.mat_size**0.5)
    weightC += self.masking(weightC)
    weightC = torch.softmax(weightC,dim = -1)
    embedC = torch.matmul(weightC,valueC)

    weightD = torch.matmul(queryD,keyD.transpose(-2,-1))
    weightD = weightD/(self.mat_size**0.5)
    weightD += self.masking(weightD)
    weightD = torch.softmax(weightD,dim = -1)
    embedD = torch.matmul(weightD,valueD)

    final_embed = torch.cat([embedA,embedB,embedC,embedD],dim = -1)

    final_embed = torch.matmul(final_embed,self.final_weight)


    return final_embed

  def cross_attention(self,input,output):
    #MULTI ATTENTION
    queryA = torch.matmul(output, self.querym1) # 1,1,64 64,16    1,16,1
    keyA =  torch.matmul(input, self.keym1) # 1,50,64 , 64,16 50,16
    valueA = torch.matmul(input, self.valuem1)

    queryB = torch.matmul(output, self.querym2)
    keyB =  torch.matmul(input, self.keym2)
    valueB = torch.matmul(input, self.valuem2)

    queryC = torch.matmul(output, self.querym3)
    keyC =  torch.matmul(input, self.keym3)
    valueC = torch.matmul(input, self.valuem3)

    queryD = torch.matmul(output, self.querym4)
    keyD =  torch.matmul(input, self.keym4)
    valueD = torch.matmul(input, self.valuem4)


    weightA = torch.matmul(queryA,keyA.transpose(-2,-1))
    weightA = weightA/(self.mat_size**0.5)
    weightA = torch.softmax(weightA,dim = -1)
    embedA = torch.matmul(weightA,valueA)


    weightB = torch.matmul(queryB,keyB.transpose(-2,-1))
    weightB = weightB/(self.mat_size**0.5)
    weightB = torch.softmax(weightB,dim = -1)
    embedB = torch.matmul(weightB,valueB)

    weightC = torch.matmul(queryC,keyC.transpose(-2,-1))
    weightC = weightC/(self.mat_size**0.5)
    weightC = torch.softmax(weightC,dim = -1)
    embedC = torch.matmul(weightC,valueC)

    weightD = torch.matmul(queryD,keyD.transpose(-2,-1))
    weightD = weightD/(self.mat_size**0.5)
    weightD = torch.softmax(weightD,dim = -1)
    embedD = torch.matmul(weightD,valueD)


    final_embed = torch.cat([embedA,embedB,embedC,embedD],dim = -1)

    final_embed = torch.matmul(final_embed,self.final_weight)

    return final_embed

  def forward(self,input,output):

    masking = self.masked_multihead(output)

    add1 = masking + output

    layernorm1 = self.layernorm1(add1)

    cross_attn = self.cross_attention(input,output)

    add2 = cross_attn + layernorm1

    layernorm2 = self.layernorm2(add2)

    linear = self.linear(layernorm2)

    add3 = layernorm2 + linear

    layernorm3 = self.layernorm3(add3)

    return layernorm3

In [3]:
class Tmr(nn.Module):
  def __init__(self,embed_dim,vocab):
    super().__init__()
    self.embed_dim = embed_dim

    self.encoder1 = Encoder(embed_dim)
    self.encoder2 = Encoder(embed_dim)
    self.encoder3 = Encoder(embed_dim)
    self.encoder4 = Encoder(embed_dim)


    self.decoder1 = Decoder(embed_dim)
    self.decoder2 = Decoder(embed_dim)
    self.decoder3 = Decoder(embed_dim)
    self.decoder4 = Decoder(embed_dim)


    self.e = vocab[0]
    self.m = vocab[1]

    self.eng = nn.Embedding(self.e,self.embed_dim)
    self.mar = nn.Embedding(self.m,self.embed_dim)


    self.final_prob = nn.Sequential(
        nn.Linear(self.embed_dim,self.m)
        #not using softmax due to use of CEloss
    )

  def positional_encodding(self,embedding):
    positional_encodding = []

    sentence = int(embedding.size(0))
    elements = int(embedding.size(1))
    items = int(embedding.size(2)/2)

    for s in range(sentence):
      for element in range(elements):
        for i in range(items):

          sine = sin((element)/(10000**(2*i/self.embed_dim)))
          positional_encodding.append(sine)
          cosine = cos((element)/(10000**(2*i/self.embed_dim)))
          positional_encodding.append(cosine)


    positional_encodding = torch.tensor(positional_encodding)
    positional_encodding = positional_encodding.reshape(sentence,element+1,items*2).to(device)

    return positional_encodding


  def embedding(self,x,y):

    english = self.eng(x)
    marathi = self.mar(y)

    return english,marathi

  def forward(self,input,output):

    input += self.positional_encodding(input)
    output += self.positional_encodding(output)

    e = self.encoder1(input)
    e = self.encoder2(e)
    #e = self.encoder3(e)
    #e = self.encoder4(e)


    d = self.decoder1(e,output)
    d = self.decoder2(e,d)
    #d = self.decoder3(e,d)
    #d = self.decoder4(e,d)


    final_output = self.final_prob(d)

    return final_output



In [4]:
class tokenize:
  def __init__(self,file,limit):
    if limit == "max":
      limit = len(file)
    self.limit = limit
    self.english = file['English']
    self.marathi = file['Marathi']
    self.mar_dict = {"<sos>":0,"<eos>":1,"<null>":2}
    self.eng_dict = {"<sos>":0,"<eos>":1,"<null>":2}
    print("limit = ",limit)

  def processing(self):
    for i,j in zip(self.english,self.marathi):
      i,j = i.replace("!","") , j.replace("!","")
      i,j = i.replace("?","") , j.replace("?","")
      i,j = i.replace(".","") , j.replace(".","")
      i,j = i.replace("'","") , j.replace("'","")
      i = i.lower()

      ekeys = i.split()
      mkeys = j.split()

      for i,j in zip(ekeys,mkeys):
        if i not in self.eng_dict:
          self.eng_dict[i] = len(self.eng_dict)

        if j not in self.mar_dict:
          self.mar_dict[j] = len(self.mar_dict)

    return self.eng_dict , self.mar_dict

  def get_dict(self):
    return self.eng_dict,self.mar_dict

  def dict_size(self):
    return len(self.eng_dict) , len(self.mar_dict)

  def inf_w2n(self,i):
    i = i.replace("!","").replace("?","").replace(".","").replace("'","")
    i = i.lower()

    new = i.split()

    start = [self.eng_dict["<sos>"]]

    value = [self.eng_dict[e] for e in new if e in self.eng_dict]

    end = [self.eng_dict["<eos>"]]

    sent = start + value + end
    while(len(sent)<50):
        sent.append(self.eng_dict["<null>"])

    return sent


  def word_to_num(self):

    eng_item = []
    mar_item = []

    for i,j in zip(self.english,self.marathi):
      if len(eng_item) > self.limit:
        break
      i,j = i.replace("!","") , j.replace("!","")
      i,j = i.replace("?","") , j.replace("?","")
      i,j = i.replace(".","") , j.replace(".","")
      i,j = i.replace("'","") , j.replace("'","")
      i = i.lower()
      eng = i.split()
      mar = j.split()

      #main conversion
      eng_num = [self.eng_dict[val] for val in eng if val in self.eng_dict]
      mar_num = [self.mar_dict[val] for val in mar if val in self.mar_dict]

      evalue = [self.eng_dict["<sos>"]]
      evalue += (eng_num)
      evalue.append(self.eng_dict["<eos>"])

      mvalue = [self.mar_dict["<sos>"]]
      mvalue += (mar_num)
      mvalue.append(self.mar_dict["<eos>"])

      while(len(evalue)<50):
        evalue.append(self.eng_dict["<null>"])
      while(len(mvalue)<50):
        mvalue.append(self.mar_dict["<null>"])

      eng_item.append(evalue)
      mar_item.append(mvalue)

    final_eng = torch.tensor(eng_item)
    final_mar = torch.tensor(mar_item)


    return final_eng,final_mar



In [5]:
from torch.utils.data import DataLoader,Dataset

class MyDataset(Dataset):
  def __init__(self,eng,mar):
    self.eng = eng
    self.mar = mar
  def __len__(self):
    return len(self.mar)
  def __getitem__(self,index):

    return self.eng[index],self.mar[index]

In [6]:
data = pd.read_csv(r"dataset\Transformer-data.csv")


#working with data
data['English'][len(data)-1]

data.tail(10)

,Unnamed: 0,English,Marathi
118954,118954,You can't view Flash content on an iPad. Howev...,तुम्ही iPad वर फ्लॅश सामग्री पाहू शकत नाही. तथ...
118955,118955,A mistake young people often make is to start ...,एकाच वेळी अनेक भाषा शिकणे सुरू करणे ही तरुणांच...
118956,118956,No matter how much you try to convince people ...,चॉकलेट हे व्हॅनिला आहे हे तुम्ही लोकांना पटवून...
118957,118957,"In 1969, Roger Miller recorded a song called ""...","1969 मध्ये रॉजर मिलरने ""यू डोन्ट वॉन्ट माय लव्..."
118958,118958,A child who is a native speaker usually knows ...,मूळ भाषिक असलेल्या मुलाला सहसा त्याच्या भाषेबद...
118959,118959,There are four main causes of alcohol-related ...,अल्कोहोल-संबंधित मृत्यूची चार मुख्य कारणे आहेत...
118960,118960,There are mothers and fathers who will lie awa...,असे आई आणि वडील आहेत जे मुले झोपी गेल्यानंतर ज...
118961,118961,A carbon footprint is the amount of carbon dio...,कार्बन फूटप्रिंट म्हणजे कार्बन डायऑक्साइड प्रद...
118962,118962,Since there are usually multiple websites on a...,कोणत्याही दिलेल्या विषयावर सहसा अनेक वेबसाइट्स...
118963,118963,"If you want to sound like a native speaker, yo...","जर तुम्हाला मूळ स्पीकरसारखा आवाज द्यायचा असेल,..."


In [11]:
token = tokenize(data,1000)

a,b = token.processing()


limit =  1000


In [12]:
lst = []
for i in data["English"]:
  i = i.split()

  lst.append(len(i))
print(max(lst))


token.dict_size()

a = token.inf_w2n("She said that she would follow him no matter where he went.")
print(a)

47
[0, 191, 413, 240, 191, 1825, 234, 116, 19, 2049, 533, 26, 641, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [14]:
from sklearn.model_selection import train_test_split

final_eng,final_mar = token.word_to_num()


X_train , X_test , y_train , y_test = train_test_split(final_eng,final_mar,train_size = 0.80)

train = MyDataset(X_train,y_train)
test = MyDataset(X_test,y_test)

train_loader = DataLoader(train,batch_size=32,shuffle = True)
test_loader = DataLoader(test,batch_size=32,shuffle = True)


In [15]:
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

vocab = token.dict_size()
model = Tmr(64,vocab).to(device)

model.train()
epoch = 10
lr = 0.0001

criterion = nn.CrossEntropyLoss(ignore_index=2)

optimizer = optim.Adam(model.parameters(),lr = lr)

for e in range(epoch):

  total_loss = 0

  print("epoch --> ",e+1)
  for eng,mar in train_loader:

    eng = eng.to(device)
    mar = mar.to(device)

    eng,mar_input = model.embedding(eng[:,:-1],mar[:,:-1])
    #print(eng.shape,mar_input.shape)

    output = model(eng,mar_input)
    optimizer.zero_grad()

    loss = criterion(output.reshape(-1,vocab[1]),mar[:,1:].reshape(-1))

    loss.backward()
    optimizer.step()

    total_loss +=loss.item()


  print(f"Loss = {total_loss}")


cuda
epoch -->  1
Loss = 250.31341171264648
epoch -->  2
Loss = 230.49797916412354
epoch -->  3
Loss = 218.95401668548584
epoch -->  4
Loss = 210.5516185760498
epoch -->  5
Loss = 203.220046043396
epoch -->  6
Loss = 196.21589469909668
epoch -->  7
Loss = 189.32451057434082
epoch -->  8
Loss = 182.4755630493164
epoch -->  9
Loss = 175.72041368484497
epoch -->  10
Loss = 169.09617137908936


In [18]:
torch.save(model.state_dict(),r"saved_model\Tws.pth")

In [44]:
vocab = token.dict_size()
model = Tmr(64,vocab)

model.load_state_dict(torch.load(r"saved_model\Tws.pth",map_location="cpu"))

model = model

In [45]:
"""You can't view Flash content on an iPad. However, you can easily email yourself the URLs of these web pages and view that content on your regular computer when you get home."""

"You can't view Flash content on an iPad. However, you can easily email yourself the URLs of these web pages and view that content on your regular computer when you get home."

In [46]:
prediction_val = 0
for e,m in train_loader:
  prediction_val = e

In [60]:
print(prediction_val[0])

tensor([  0, 137,  30,   1,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2,   2,   2,   2,   2,   2,   2,   2])


In [65]:
model.eval()
eng_dict,mar_dict = token.get_dict()

device = next(model.parameters()).device
start = "<sos>"
mar_input = [mar_dict[start]]

user = "hello"

user = user.replace("!","").replace("?","").replace(".","").replace("'","").lower()

user = token.inf_w2n(user)

user_input = torch.tensor(user).reshape(1,-1)

with torch.no_grad():
  for i in range(1):
    initial = torch.tensor(mar_input)
    #a = input("enter sentence - ")
    a,b = model.embedding(user_input,initial)
    b = b.reshape(1,-1,64)
    #print(a.shape)
    #print(b.shape)
    out = model(a.reshape(1,50,-1),b)

    generated = out[0,-1,:].argmax(-1).item()
    val = torch.argmax(out)
    if(generated == token.mar_dict["<eos>"]):
      break

    mar_input.append(generated)
  print(mar_input,len(mar_input))


last_logits = out[0, -1, :]

top_values, top_ids = torch.topk(last_logits, 10)

for idx, score in zip(top_ids.tolist(), top_values.tolist()):
    word = next(
        (w for w, v in mar_dict.items() if v == idx),
        "<unknown>"
    )
    print(f"{word}: {score:.4f}")

[0] 1
<eos>: 5.9827
टॉम: 2.8675
मी: 2.4157
मलाच: 2.3159
राहणे: 2.2359
आहे: 2.0729
शकतो: 1.9304
फोन: 1.9160
केले: 1.8929
वेगवान: 1.8409
